# 00 — Installer les bibliothèques (Python 3.9)

**Sans environnement virtuel et sans Makefile.** Les installations s’effectuent dans le Python du noyau Jupyter actuellement sélectionné.

⚠️ Modifier l’installation Python existante peut affecter d’autres projets. Crée une sauvegarde de tes versions si nécessaire. Aucune cellule n’installe automatiquement sans ton accord.

### Étape 1 — Confirmer le bon noyau Python 3.9
Dans VS Code, sélectionner un noyau **Python 3.9** en haut à droite. L’installation ne crée aucun environnement virtuel.


In [2]:
import sys,subprocess,platform,importlib.util
from pathlib import Path
print('Version :',sys.version)
print('Interpreteur :',sys.executable)
print('Windows :',platform.platform())
if sys.version_info[:2]!=(3,9):
    raise RuntimeError('Selectionne ton Python 3.9 comme noyau Jupyter avant d installer. Version detectee : '+sys.version.split()[0])
print('OK : Python 3.9 selectionne.')


Version : 3.9.13 (tags/v3.9.13:6de2ca5, May 17 2022, 16:36:42) [MSC v.1929 64 bit (AMD64)]
Interpreteur : c:\Users\elkat\AppData\Local\Programs\Python\Python39\python.exe
Windows : Windows-10-10.0.19045-SP0
OK : Python 3.9 selectionne.


### Étape 2 — Choisir ce que tu veux installer
**Base** : Jupyter, données, graphiques, caméra et diagnostic. **IA** : PyTorch, Stable-Baselines3, Gymnasium, TensorBoard. **Optionnel** : bibliothèques héritées/avancées.

Le paquet `gym==0.21` de l’ancien Assetto Corsa Gym n’est **pas** installé automatiquement : ses métadonnées et ses anciens outils de build peuvent entrer en conflit avec le nouveau stack. Les notebooks utilisent directement le protocole du plugin, sans dépendre de `gym==0.21`.


In [3]:
# SEULE CELLULE A MODIFIER POUR INSTALLER :
INSTALLER_BASE=False
INSTALLER_IA=False
INSTALLER_OPTIONS=False
# 'cpu' ou 'cu121' (GPU NVIDIA avec pilotes compatibles); ne pas choisir au hasard
PYTORCH_MODE='cpu'
# Ne jamais executer ces installations automatiquement a l ouverture du notebook.


In [4]:
# import sys

# print("Version :", sys.version)
# print("Python utilisé :", sys.executable)
# print("Tout fonctionne !")


In [5]:
BASE=[
    'pip==24.3.1','setuptools==75.8.2','wheel==0.45.1',
    'ipykernel==6.29.5','jupyterlab==4.2.5','notebook==7.2.2','ipywidgets==8.1.5',
    'numpy==1.23.5','pandas==1.5.3','scipy==1.10.1','matplotlib==3.7.5',
    'Pillow==10.4.0','PyYAML==6.0.2','tqdm==4.66.5',
    'opencv-python==4.10.0.84','psutil==6.0.0','imageio==2.35.1',
    'requests==2.32.3','packaging==24.1','pywin32==306',
]
IA=[
    'torch==2.1.2','torchvision==0.16.2',
    'gymnasium==0.29.1','stable-baselines3==2.2.1',
    'tensorboard==2.17.1',
]
OPTIONS=[
    'pygame==2.6.1','pyglet==1.5.27','omegaconf==2.3.0',
    'numba==0.60.0','pyarrow==17.0.0','scikit-optimize==0.10.2',
    'termcolor==2.4.0','python-box==7.2.0',
]
print('Base :',len(BASE),'paquets; IA :',len(IA),'paquets; Options :',len(OPTIONS),'paquets')


Base : 20 paquets; IA : 5 paquets; Options : 8 paquets


In [6]:
def pip_install(packages,extra_args=None):
    command=[sys.executable,'-m','pip','install']+list(extra_args or [])+list(packages)
    print('Execution :', ' '.join(command))
    subprocess.run(command,check=True)
if INSTALLER_BASE:
    # Les mises a jour du noyau Jupyter peuvent necessiter un redemarrage.
    pip_install(BASE)
else:
    print('Installation BASE non demandee (INSTALLER_BASE=False).')


Installation BASE non demandee (INSTALLER_BASE=False).


In [7]:
if INSTALLER_IA:
    if PYTORCH_MODE not in ('cpu','cu121'):
        raise ValueError('PYTORCH_MODE doit valoir cpu ou cu121.')
    pytorch_index='https://download.pytorch.org/whl/'+PYTORCH_MODE
    pip_install(IA[:2],['--index-url',pytorch_index])
    pip_install(IA[2:])
else:
    print('Installation IA non demandee (INSTALLER_IA=False).')


Installation IA non demandee (INSTALLER_IA=False).


In [8]:
if INSTALLER_OPTIONS:
    pip_install(OPTIONS)
else:
    print('Options non installees.')


Options non installees.


### Étape 3 — Contrôler les importations
Redémarrer le noyau après les installations, puis exécuter :


In [9]:
import importlib.util
PACKAGES={'numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib','cv2':'opencv-python',
          'PIL':'Pillow','scipy':'scipy','yaml':'PyYAML','torch':'torch',
          'gymnasium':'gymnasium','stable_baselines3':'stable-baselines3',
          'tensorboard':'tensorboard','ipykernel':'ipykernel','win32event':'pywin32'}
for mod,package in PACKAGES.items():
    print(('[OK] ' if importlib.util.find_spec(mod) else '[MANQUANT] ')+package)
try:
    import torch
    print('Torch :',torch.__version__,'CUDA active :',torch.cuda.is_available())
except ImportError:print('Torch non installe.')


[OK] numpy
[OK] pandas
[OK] matplotlib
[MANQUANT] opencv-python
[OK] Pillow
[OK] scipy
[OK] PyYAML
[OK] torch
[MANQUANT] gymnasium
[MANQUANT] stable-baselines3
[OK] tensorboard
[OK] ipykernel
[OK] pywin32


c:\Users\elkat\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch : 1.12.1+cpu CUDA active : False


### À savoir
- `ctypes`, `socket`, `time`, `json`, `pathlib` sont dans Python, rien à installer.
- vJoy nécessite son **pilote Windows**, une DLL et sa configuration dans Assetto Corsa, pas un paquet pip.
- `assetto_corsa_gym` est **déjà présent dans ton dépôt** et n’est pas dupliqué.
- Le runtime Python interne du plugin Assetto Corsa est distinct du noyau Jupyter.
- Python 3.9 est en fin de vie : utilise-le seulement pour ce projet si tu dois conserver la compatibilité.
